# 05 — Evaluation report

**Read this one end to end.** It is the consolidated evidence for every
model in Phase 2, and it is written to be read rather than skimmed.

Four rules the numbers below obey:

1. **Every metric carries its split strategy.** 'F1 0.78' is not a result;
   'F1 0.78, grouped by claim, 5-fold, ±0.04' is.
2. **Accuracy is banned.** On an imbalanced target it rewards predicting the
   majority class. PR-AUC leads, because it degrades exactly when the model
   starts crying wolf.
3. **Every model sits next to its baselines.** If the fine-tune does not
   clear TF-IDF, that is stated, not buried.
4. **Confidence intervals decide what counts as a difference.** A 2-point F1
   gap on a 500-row test set is noise, and overlapping intervals are reported
   as 'not separable', never as a win.

---

## The honest summary, up front

Every benchmark this project uses is access-gated. If the tables below are
stamped **DEMO FIXTURE**, they were computed on committed synthetic data
that reproduces each dataset's shape and none of its content. Those numbers
demonstrate that the training and evaluation path executes; they are not
results and must not be cited as any.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modeling.config import get_settings, run_fingerprint, set_all_seeds
from modeling.io import CorpusReader, ScoredStore

set_all_seeds()
settings = get_settings()
reader = CorpusReader(settings)
store = ScoredStore(settings)

# Every number below is tied to this fingerprint. If a rerun disagrees with
# a committed figure, this block is where the diagnosis starts.
fingerprint = run_fingerprint()
print(f"seed={fingerprint['seed']}  device={fingerprint['device']}  "
      f"corpus={fingerprint['input_manifest_hash']}")

In [ ]:
import json

rows = []
for module_dir in sorted(p for p in settings.eval_dir.iterdir() if p.is_dir()):
    for version_dir in sorted(p for p in module_dir.iterdir() if p.is_dir()):
        path = version_dir / 'metrics.json'
        if not path.exists():
            continue
        payload = json.loads(path.read_text())
        metrics = payload['metrics']
        rows.append({
            'module': module_dir.name,
            'version': version_dir.name,
            'demo': metrics.get('is_demo'),
            'n_test': metrics['n_test'],
            'macro_F1': metrics['macro_f1']['value'],
            'F1_ci': f"[{metrics['macro_f1']['ci_low']:.3f}, {metrics['macro_f1']['ci_high']:.3f}]",
            'PR_AUC': (metrics['pr_auc'] or {}).get('value'),
            'Brier': metrics.get('brier'),
            'split': metrics['split'],
        })

summary = pd.DataFrame(rows)
summary if len(summary) else print('No eval artifacts. Run: python -m modeling.cli train misinfo --demo')

## Baselines: what did the expensive model buy?

A baseline is not there to be beaten. It is there to answer the question
above, and sometimes the answer is 'nothing' — which is worth more than a
tuned number, and is reported here whichever way it comes out.

In [ ]:
for module_dir in sorted(p for p in settings.eval_dir.iterdir() if p.is_dir()):
    for version_dir in sorted(p for p in module_dir.iterdir() if p.is_dir()):
        path = version_dir / 'metrics.json'
        if not path.exists():
            continue
        payload = json.loads(path.read_text())
        baselines = payload.get('baselines')
        if not baselines:
            continue
        print(f"### {module_dir.name} ({version_dir.name})")
        for name, verdict in baselines['verdicts'].items():
            print(f"  {name:24} delta {verdict['delta']:+.3f}  {verdict['verdict']}")
        if not baselines['clears_every_baseline']:
            print('  >> does NOT cleanly clear every baseline — see the report')
        print()

## Calibration

Phase 4 multiplies these scores together into one risk number. That is only
meaningful if each input is a calibrated probability — if `misinfo_prob=0.7`
genuinely means 'about 70% of records scored 0.7 are misinformation-like'.

Note the fallback: isotonic regression below 200 validation rows fits a step
function to noise and produces confident 0.0/1.0 outputs, so it falls back to
Platt scaling and says so. If calibration made the Brier score *worse*, that
appears here too.

In [ ]:
for module_dir in sorted(p for p in settings.eval_dir.iterdir() if p.is_dir()):
    for version_dir in sorted(p for p in module_dir.iterdir() if p.is_dir()):
        path = version_dir / 'metrics.json'
        if not path.exists():
            continue
        calibration = json.loads(path.read_text()).get('calibration')
        if not calibration:
            continue
        print(f"{module_dir.name}: {calibration['method']} on "
              f"{calibration['n_calibration']} rows, Brier "
              f"{calibration['brier_before']:.4f} -> {calibration['brier_after']:.4f}")
        if calibration.get('note'):
            print(f"  note: {calibration['note']}")
        curve = calibration['reliability_after']
        if curve['predicted']:
            plt.figure(figsize=(4, 4))
            plt.plot([0, 1], [0, 1], '--', color='grey', label='perfect')
            plt.plot(calibration['reliability_before']['predicted'],
                     calibration['reliability_before']['observed'], 'o-', label='before')
            plt.plot(curve['predicted'], curve['observed'], 'o-', label='after')
            plt.xlabel('predicted'); plt.ylabel('observed')
            plt.title(f'{module_dir.name} reliability'); plt.legend(); plt.tight_layout()
            plt.show()

## Ablation

> **The fusion used here is provisional and for measurement only.** Phase 4
> owns the product's 0–100 risk score, deliberately, because the weighting is
> a documented product decision rather than a model output. What is used below
> is an equally-weighted, null-aware mean of the available components — it
> exists so that 'adding coordination changed the ranking by this much' is a
> sentence with a number in it, and for no other purpose.

Read the coverage table underneath. A configuration that adds a component
present on 0% of narratives is identical to the row above it, and that is
information rather than a bug.

In [ ]:
from modeling.eval.ablation import run_ablation, write_ablation

ablation = run_ablation()
print(ablation.render())
for path in write_ablation(ablation):
    print('wrote', path)

## Error analysis

A confusion matrix says *how many* the model got wrong. It never says *what
kind* of wrong, and the kind is what decides whether a model is deployable.

The category counts are a keyword-and-shape triage pass, not the analysis.
The analysis is the prose in `artifacts/error_analysis/<module>.md`, written
after reading the uncategorized examples.

In [ ]:
for path in sorted(settings.error_analysis_dir.glob('*.md')):
    print('=' * 70)
    print(path.name)
    print('=' * 70)
    print(path.read_text()[:2500])
    print()

## Fairness check

Score distributions across the corpus's language groups and across a topical
slice, for the toxicity and misinformation classifiers.

You do not need to fix what this finds. You do need to report it. The
toxicity model in particular is known to over-flag African-American English
and identity terms in non-pejorative use — that is a property of the Jigsaw
annotations it learned from, it is documented in the model card, and it means
`toxicity` must never be read as 'this account is abusive'.

In [ ]:
scores = store.read('record_scores')
records = reader.records(columns=['id', 'source', 'lang'])
if len(scores) and len(records):
    joined = records.merge(scores, left_on='id', right_on='record_id', how='inner')
    from modeling.eval.metrics import group_slice_report, score_distribution
    for column in ['toxicity', 'misinfo_prob']:
        if column not in joined or joined[column].notna().sum() == 0:
            continue
        print(f'--- {column} ---')
        print('overall:', score_distribution(joined[column].to_numpy())['quantiles'])
        print('by language:', group_slice_report(joined[column].to_numpy(),
                                                 joined['lang'].fillna('unknown').to_numpy()))
        print('by source:  ', group_slice_report(joined[column].to_numpy(),
                                                 joined['source'].to_numpy()))
        print()

## What these models do not tell you

Stated here because it is the part most easily lost between a notebook and a
dashboard.

- **Benchmark performance is not production performance.** LIAR is
  politicians' statements; FakeNewsNet is news headlines; this corpus is
  Reddit comments, Mastodon toots and GDELT article metadata. Expect a large
  drop. The measured transfer gap — 100 hand-labelled corpus records — is the
  only number that describes production behaviour.
- **A high `misinfo_prob` is not a determination that a claim is false.** It
  is a similarity judgement to patterns in fact-checked corpora.
- **A high `bot_prob` is not a determination that an account is automated.**
  The training data is Twitter; this corpus is not.
- **`anomaly_score` is a within-corpus rank, not a probability.** It cannot be
  multiplied into a probability product.
- **A null is not a zero.** Every null in the scored tables means 'not
  assessed' and carries a reason code.